# Using OpenDP Synth - test

## Step 1: Install the library

It can be installed via the pip command:

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..')))
# !pip install lomas_client

In [ ]:
from lomas_client import Client
import numpy as np
import opendp.prelude as dp

## Step 2: Initialise the client

Once the library is installed, a Client object must be created. It is responsible for sending sending requests to the server and processing responses in the local environment. It enables a seamless interaction with the server. 

The client needs a few parameters to be created. Usually, these would be set in the environment by the system administrator and be transparent to lomas users. In this instance, the following code snippet sets a few of these parameters that are specific to this notebook. 

In [ ]:
# The following would usually be set in the environment by a system administrator
# and be tranparent to lomas users.
# Uncomment them if you are running against a Kubernetes deployment.
# They have already been set for you if you are running locally within a devenv or the Jupyter lab set up by Docker compose.

import os
# os.environ["LOMAS_CLIENT_APP_URL"] = "https://lomas.example.com:443"
# os.environ["LOMAS_CLIENT_OIDC_DISCOVERY_URL"] = "https://dex.example.com:443/.well-known/openid-configuration"
# os.environ["LOMAS_CLIENT_TELEMETRY__ENABLED"] = "false"
# os.environ["LOMAS_CLIENT_TELEMETRY__COLLECTOR_ENDPOINT"] = "http://otel.example.com:445"
# os.environ["LOMAS_CLIENT_TELEMETRY__COLLECTOR_INSECURE"] = "true"
# os.environ["LOMAS_CLIENT_TELEMETRY__SERVICE_ID"] = "my-app-client"
# os.environ["LOMAS_CLIENT_REALM"] = "lomas"

# We set these ones because they are specific to this notebook.

os.environ["LOMAS_CLIENT_USER_NAME"] = "mr.corona@example.com"
os.environ["LOMAS_CLIENT_USER_PASSWORD"] = "mr.corona"
os.environ["LOMAS_CLIENT_DATASET_NAME"] = "COVID_SYNTHETIC"

# Note that all client settings can also be passed as keyword arguments to the Client constructor.
# eg. client = Client(user_name = "Mr.corona") takes precedence over setting the "LOMAS_CLIENT_USER_NAME"
# environment variable.

In [ ]:
client = Client()

## Step 3: Metadata and dummy dataset

### Getting dataset metadata

The user has never seen the data and as a first step to understand what is available to her, she would like to check the metadata of the dataset. Therefore, she just needs to call the `get_dataset_metadata()` function of the client. As this is public information, this does not cost any budget.

This function returns metadata information in a format based on [SmartnoiseSQL dictionary format](https://docs.smartnoise.org/sql/metadata.html#dictionary-format), where among other, there is information about all the available columns, their type, bound values (see Smartnoise page for more details). Any metadata is required for Smartnoise-SQL is also required here and additional information such that the different categories in a string type column column can be added.

In [ ]:
covid_metadata = client.get_dataset_metadata()
covid_metadata

### Get a dummy dataset

Now, that she has seen and understood the metadata, she wants to get an even better understanding of the dataset (but is still not able to see it). A solution to have an idea of what the dataset looks like it to create a dummy dataset. 

Based on the public metadata of the dataset, a random dataframe can be created created. By default, there will be 100 rows and the seed is set to 42 to ensure reproducibility, but these 2 variables can be changed to obtain different dummy datasets.
Getting a dummy dataset does not affect the budget as there is no differential privacy here. It is not a synthetic dataset and all that could be learn here is already present in the public metadata (it is created randomly on the fly based on the metadata).

Dr. FSO first create a dummy dataset with 200 rows and chooses a seed of 0.

In [ ]:
columns = ['country', 'subType', 'hospitalization', 'death', "temporal", "date"]
res = client.opendp_synth.query(epsilon=100.0, delta=0.001, columns=columns)
print(res)

epsilon=0.0 delta=0.0 requested_by='Mr.Corona' result=OpenDPPolarsQueryResult(type=<DPLibraries.OPENDP_POLARS: 'opendp_polars'>, value=shape: (50_064, 6)
┌─────────┬─────────┬─────────────────┬───────┬──────────┬─────────────────────┐
│ country ┆ subType ┆ hospitalization ┆ death ┆ temporal ┆ date                │
│ ---     ┆ ---     ┆ ---             ┆ ---   ┆ ---      ┆ ---                 │
│ str     ┆ str     ┆ bool            ┆ bool  ┆ i64      ┆ str                 │
╞═════════╪═════════╪═════════════════╪═══════╪══════════╪═════════════════════╡
│ CH      ┆ BA.1    ┆ true            ┆ true  ┆ 6        ┆ 2022-09-06 00:00:00 │
│ CH      ┆ BA.1    ┆ true            ┆ true  ┆ 6        ┆ 2022-09-06 00:00:00 │
│ CH      ┆ BA.1    ┆ true            ┆ false ┆ 6        ┆ 2022-09-06 00:00:00 │
│ CH      ┆ BA.1    ┆ true            ┆ false ┆ 6        ┆ 2022-09-06 00:00:00 │
│ CH      ┆ BA.1    ┆ true            ┆ false ┆ 6        ┆ 2022-09-06 00:00:00 │
│ …       ┆ …       ┆ …             

In [ ]:
res.result.value.describe()

statistic,country,subType,hospitalization,death,temporal,date
str,str,str,f64,f64,f64,str
"""count""","""50059""","""17156""",50062.0,50062.0,50064.0,"""45426"""
"""null_count""","""5""","""32908""",2.0,2.0,0.0,"""4638"""
"""mean""",null,null,0.007331,0.000759,26.711909,null
"""std""",null,null,null,null,17.959332,null
"""min""","""CH""","""BA.1""",0.0,0.0,6.0,"""2022-09-06 00:00:00"""
"""25%""",null,null,null,null,7.0,null
"""50%""",null,null,null,null,28.0,null
"""75%""",null,null,null,null,46.0,null
"""max""","""unknown""","""unknown""",1.0,1.0,48.0,"""2023-04-16 00:00:00"""


In [ ]:
import polars as pl
df = pl.scan_csv("/home/lancelot/dsccadminch/lomas/server/data/datasets/covid_synthetic_data.csv")

In [ ]:
df.select(columns).collect().describe()

statistic,country,subType,hospitalization,death,temporal,date
str,str,str,f64,f64,f64,str
"""count""","""50048""","""50048""",50048.0,50048.0,50048.0,"""50048"""
"""null_count""","""0""","""0""",0.0,0.0,0.0,"""0"""
"""mean""",null,null,0.006973,0.000759,26.521759,null
"""std""",null,null,null,null,19.212237,null
"""min""","""CH""","""BA.1""",0.0,0.0,1.0,"""2022-08-01"""
"""25%""",null,null,null,null,7.0,null
"""50%""",null,null,null,null,31.0,null
"""75%""",null,null,null,null,46.0,null
"""max""","""unknown""","""unknown""",1.0,1.0,52.0,"""2023-07-30"""


In [ ]:
# THIS will FAIL because all keys and cuts for those columns is given
# Need to use delta = 0 in that case
columns = ['country', 'subType', 'hospitalization', 'death', "temporal"]
res = client.opendp_synth.query(epsilon=100.0, delta=0.001, columns=columns)
print(res)

LomasAPIException: Exception from <DPLibraries.OPENDP_SYNTH: 'opendp_synth'> library: Error releasing synthetic data:delta (0.001) must be zero because keys and cuts span all columns

In [ ]:
# WORKING
columns = ['country', 'subType', 'hospitalization', 'death', "temporal"]
res = client.opendp_synth.query(epsilon=100.0, columns=columns)
print(res)

epsilon=0.0 delta=0.0 requested_by='Mr.Corona' result=OpenDPPolarsQueryResult(type=<DPLibraries.OPENDP_POLARS: 'opendp_polars'>, value=shape: (50_056, 5)
┌─────────┬─────────┬─────────────────┬───────┬──────────┐
│ country ┆ subType ┆ hospitalization ┆ death ┆ temporal │
│ ---     ┆ ---     ┆ ---             ┆ ---   ┆ ---      │
│ str     ┆ str     ┆ bool            ┆ bool  ┆ i64      │
╞═════════╪═════════╪═════════════════╪═══════╪══════════╡
│ CH      ┆ BA.1    ┆ true            ┆ true  ┆ 6        │
│ CH      ┆ BA.1    ┆ true            ┆ true  ┆ 6        │
│ FL      ┆ BA.1    ┆ true            ┆ false ┆ 6        │
│ CH      ┆ BA.1    ┆ true            ┆ false ┆ 6        │
│ CH      ┆ BA.1    ┆ true            ┆ false ┆ 6        │
│ …       ┆ …       ┆ …               ┆ …     ┆ …        │
│ CH      ┆ null    ┆ false           ┆ null  ┆ 48       │
│ CH      ┆ BA.1    ┆ null            ┆ false ┆ 48       │
│ CH      ┆ BQ.1    ┆ null            ┆ false ┆ 48       │
│ CH      ┆ unknown 

# test TITANIC

In [ ]:
import os
# os.environ["LOMAS_CLIENT_APP_URL"] = "https://lomas.example.com:443"
# os.environ["LOMAS_CLIENT_OIDC_DISCOVERY_URL"] = "https://dex.example.com:443/.well-known/openid-configuration"
# os.environ["LOMAS_CLIENT_TELEMETRY__ENABLED"] = "false"
# os.environ["LOMAS_CLIENT_TELEMETRY__COLLECTOR_ENDPOINT"] = "http://otel.example.com:445"
# os.environ["LOMAS_CLIENT_TELEMETRY__COLLECTOR_INSECURE"] = "true"
# os.environ["LOMAS_CLIENT_TELEMETRY__SERVICE_ID"] = "my-app-client"
# os.environ["LOMAS_CLIENT_REALM"] = "lomas"

# We set these ones because they are specific to this notebook.

os.environ["LOMAS_CLIENT_USER_NAME"] = "jack@example.com"
os.environ["LOMAS_CLIENT_USER_PASSWORD"] = "jack"
os.environ["LOMAS_CLIENT_DATASET_NAME"] = "TITANIC"
client = Client()

In [ ]:
res = client.opendp_synth.query(epsilon=5.0, delta = 0.001)
res

QueryResponse(epsilon=0.0, delta=0.0, requested_by='Jack', result=OpenDPPolarsQueryResult(type=<DPLibraries.OPENDP_POLARS: 'opendp_polars'>, value=shape: (891, 8)
┌──────────┬────────┬─────────────────────────┬────────┬───────────┬───────┬───────┬────────────┐
│ Survived ┆ Pclass ┆ Name                    ┆ Sex    ┆ Age       ┆ SibSp ┆ Parch ┆ Fare       │
│ ---      ┆ ---    ┆ ---                     ┆ ---    ┆ ---       ┆ ---   ┆ ---   ┆ ---        │
│ bool     ┆ i64    ┆ str                     ┆ str    ┆ f64       ┆ i64   ┆ i64   ┆ f64        │
╞══════════╪════════╪═════════════════════════╪════════╪═══════════╪═══════╪═══════╪════════════╡
│ true     ┆ 3      ┆ Mrs. Juha (Maria Emilia ┆ male   ┆ 9.190488  ┆ 1     ┆ 2     ┆ 99.529784  │
│          ┆        ┆ Ojala)…                 ┆        ┆           ┆       ┆       ┆            │
│ true     ┆ 3      ┆ Mrs. Juha (Maria Emilia ┆ male   ┆ 9.987658  ┆ 1     ┆ 2     ┆ 99.818191  │
│          ┆        ┆ Ojala)…                 ┆      